# 📑 Hoist EOM - L3869 Decisions Report
 
## 🔗 Links
- **[BRD L3869 Decisions - TBD](tbd)**  
  Comprehensive documentation outlining the business requirements for this report.
 
- **[Story Ticket - TBD](tbd)**  
  Details about the story ticket, including tasks and progress.
  
- **[UAT Findings Ticket]()**  
  A summary of findings from UAT.
 
---

In [0]:
# %pip install openpyxl
# %restart_python

In [0]:
import os
import sys

dbutils.widgets.text("run_date", "")
dbutils.widgets.text("adhoc_run", '')

try:
    notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    repo_root = "/Workspace" + notebook_path.rsplit("/HoistInvestorReports/", 1)[0] 
    sys.path.insert(0, os.path.join(repo_root, "aws_utils"))
    sys.path.insert(0, os.path.join(repo_root, "01-utils"))
except Exception:
    sys.path.append("../../aws_utils")
    sys.path.append("../../01-utils")

from aws_utils import get_env_prefix, get_env_fullname
from helpers import load_configs, build_portfolio_filter

env = get_env_prefix()

In [0]:
from datetime import datetime
from pytz import timezone
from pyspark.sql import functions as F
from pyspark.sql.window import Window

athens_tz = timezone('Europe/Athens')
adhoc_run = dbutils.widgets.get("adhoc_run")

if adhoc_run == '1':
    dbutils.widgets.text("run_date", '')
    run_date = dbutils.widgets.get("run_date")
else:
    now_date = datetime.now(athens_tz).strftime('%Y-%m-%d')
    now_date_str = datetime.now(athens_tz).strftime('%Y%m%d')
    run_date = spark.sql(f"SELECT CurrentMonthLastWorkingDate FROM `{env}-cdh`.reference.ref_calendar WHERE CalendarDate = last_day(add_months(to_date('{now_date}'), -1))").first()["CurrentMonthLastWorkingDate"]

now_datetime_str = datetime.now(athens_tz).strftime('%Y%m%dT%H%M%S')

In [0]:
from typing import Any
from delta import DeltaTable  # type: ignore
from pyspark.sql.functions import (
    coalesce,
    col,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    lit,
    sha2,
    substring,
    trim,
    when,
    months_between,
    dateadd,
    datediff,
    regexp_replace
)

from typing import Optional
import delta  # type: ignore
from pyspark.sql import SparkSession
import pandas as pd

In [0]:
## Portfolios In Scope - All HOIST portfolios
PortfolioName = "('Cell', 'Orbit', 'Pearl Portfolio A', 'Mercury', 'Light')"    

DecomissionDate = "2026-06-04"

In [0]:
reassignments = spark.sql(f"""                             
        select distinct re.LoanID
        from `{env}-reportinputfiles`.investor.hoist_reassignments as re
          """)
reassignments.createOrReplaceTempView("reassignments")

In [0]:
exposure = spark.sql(f"""
select cdi, partytaxregistrationnumber,portfolioname,'{run_date}' AS RunDate 
from (
     SELECT DISTINCT exp.cdi, exp.partytaxregistrationnumber,exp.portfolioname
     , rank()OVER (PARTITION BY exp.cdi ORDER BY exp.businessdate DESC) rnk

     FROM `{env}-cdh`.primary.exposure as exp
     LEFT JOIN reassignments as re on re.LoanID = exp.ExposureKey 
     WHERE exp.PortfolioName IN {PortfolioName}
     and exp.businessdate <= '{run_date}'
     and exp.sourcesystem = 'profits'
     aND RE.LoanID IS NULL AND (exp.JUSTIFICATIONCODE IS NULL OR exp.JUSTIFICATIONCODE NOT IN (43055,43056))
)where rnk=1
""")
exposure.createOrReplaceTempView("exposure")

In [0]:
dunderlaw = spark.sql(f"""
SELECT * FROM (          
SELECT DISTINCT
			   L.CDI
			  ,L.PortfolioName   
			  ,L.CourtDate AS HearingDate  
			  ,L.CourtOutcome
			  ,CASE WHEN L.ApplicationStatus = 'Resigned' THEN 'Resignation' ELSE L.ApplicationStatus END AS  L3869Status  
			  ,L.TypeOfCourt AS KindCourt
              ,L.BusinessDate
              ,RANK()OVER(PARTITION BY  L.CDI ORDER BY L.BusinessDate DESC) RNK
FROM `{env}-cdh`.derivative.dunderlaw L
WHERE L.BusinessDate <= '{run_date}'
AND L.PortfolioName in {PortfolioName}
AND L.UnderLawType ILIKE '%3869%'
and l.SourceSystem = 'process'
) WHERE RNK=1
      """)
dunderlaw.createOrReplaceTempView("dunderlaw")

In [0]:
dunderlaw_1 = spark.sql(f"""
SELECT * FROM (          
SELECT DISTINCT
			    dex.PartyTaxRegistrationNumber
			  ,L.PortfolioName   
			  ,L.HearingDate  
			  ,L.CourtOutcome
			  ,L.L3869Status  
			  ,L.KindCourt
              ,RANK()OVER(PARTITION BY   dex.PartyTaxRegistrationNumber,L.PortfolioName ORDER BY L.BusinessDate DESC) RNK
FROM dunderlaw L
INNER JOIN exposure dex on dex.CDI=L.CDI and L.PortfolioName=dex.PortfolioName
) WHERE RNK=1
      """)
dunderlaw_1.createOrReplaceTempView("dunderlaw_1")

In [0]:
Final = spark.sql(f"""
          
SELECT DISTINCT
			   L.PartyTaxRegistrationNumber AS AFM 
			  ,L.PortfolioName   
			  ,CASE WHEN L.HearingDate='1753-01-01' THEN NULL ELSE L.HearingDate END AS HearingDate  
			  ,L.CourtOutcome
			  ,L.L3869Status  
			  ,L.KindCourt
			  ,CAST('{run_date}' AS DATE) AS BusinessDate
FROM dunderlaw_1 L
WHERE L.PartyTaxRegistrationNumber is not null and L.PartyTaxRegistrationNumber <> ''

      """)
Final.createOrReplaceTempView("Final")

In [0]:
target_table_path_l3869decisions = f"`{env}-cdh-delivery`.investorreports.hoist_l3869decisions"

spark.sql(f"""DELETE FROM {target_table_path_l3869decisions} WHERE BusinessDate = '{run_date}' AND PortfolioName IN {PortfolioName}""")

Final = Final.withColumn("IngestionTimestamp", lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")))

Final.write \
        .format("delta") \
        .partitionBy("BusinessDate") \
        .mode("append") \
        .saveAsTable(target_table_path_l3869decisions) # in order to write in delta_lake


spark.sql(f"REPAIR TABLE {target_table_path_l3869decisions} SYNC METADATA")